# Titanic Dataset — Model Evaluation & Tuning: Beyond Accuracy
**Track:** Neurofive ML Track — Model Evaluation & Tuning
**Goal:** Go beyond accuracy to evaluate the Titanic classification model properly, then use hyperparameter tuning to systematically improve it.

This continues from the earlier classification work, using the same dataset and cleaning steps:
`https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv`


## 1. Imports

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score


## 2. Load and clean the dataset (same approach as before)

In [2]:
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

df_clean = df.copy()
df_clean['Age'] = df_clean.groupby('Pclass')['Age'].transform(lambda x: x.fillna(x.median()))
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])
df_clean['has_cabin'] = df_clean['Cabin'].notnull().astype(int)
df_clean = df_clean.drop(columns=['Cabin'])

features = df_clean.drop(columns=['PassengerId', 'Survived', 'Name', 'Ticket'])
target = df_clean['Survived']
features_encoded = pd.get_dummies(features, columns=['Sex', 'Embarked'], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    features_encoded, target, test_size=0.2, random_state=42, stratify=target
)

print("Survival rate in full dataset:", target.mean().round(3))
print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)


Survival rate in full dataset: 0.384
Train shape: (712, 9) | Test shape: (179, 9)


## 3. Baseline model (same as the original classification task)

In [3]:
baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)

baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
print(f"Baseline accuracy: {baseline_accuracy:.4f}")


Baseline accuracy: 0.8101


## 4. Precision, Recall, F1-score — going beyond accuracy

In [4]:
print(classification_report(y_test, y_pred_baseline, target_names=['Did Not Survive', 'Survived']))


                 precision    recall  f1-score   support

Did Not Survive       0.83      0.87      0.85       110
       Survived       0.78      0.71      0.74        69

       accuracy                           0.81       179
      macro avg       0.80      0.79      0.80       179
   weighted avg       0.81      0.81      0.81       179



## 5. Why accuracy alone can be misleading for imbalanced datasets

In my own words: accuracy just counts what fraction of predictions were correct overall, but it treats every mistake the same regardless of which class it came from. If a dataset is imbalanced — say 90% of passengers did not survive and only 10% did — a model that just predicts "did not survive" for everyone would score 90% accuracy while being completely useless at its actual job of identifying survivors. It would have 0% recall on the class that actually matters. Titanic's survival split (~38% survived, ~62% did not) is not wildly imbalanced, but it's imbalanced enough that accuracy alone can hide whether the model is genuinely good at spotting survivors versus just being good at spotting the majority class. Precision, recall, and F1-score break performance apart *per class*, so I can see specifically how well the model catches actual survivors (recall), how often its "survived" predictions are trustworthy (precision), and a balanced summary of both (F1) — none of which a single accuracy number can show on its own.


## 6. Hyperparameter tuning with GridSearchCV

In [5]:
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']  # liblinear supports both l1 and l2 penalties
}

grid_search = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best hyperparameters:", grid_search.best_params_)
print("Best cross-validated F1 score:", round(grid_search.best_score_, 4))


Best hyperparameters: {'C': 100, 'penalty': 'l1', 'solver': 'liblinear'}
Best cross-validated F1 score: 0.7323


## 7. Evaluate the tuned model on the test set

In [6]:
tuned_model = grid_search.best_estimator_
y_pred_tuned = tuned_model.predict(X_test)

tuned_accuracy = accuracy_score(y_test, y_pred_tuned)
print(f"Tuned model accuracy: {tuned_accuracy:.4f}")
print()
print(classification_report(y_test, y_pred_tuned, target_names=['Did Not Survive', 'Survived']))


Tuned model accuracy: 0.8101

                 precision    recall  f1-score   support

Did Not Survive       0.83      0.86      0.85       110
       Survived       0.77      0.72      0.75        69

       accuracy                           0.81       179
      macro avg       0.80      0.79      0.80       179
   weighted avg       0.81      0.81      0.81       179



## 8. Before vs. After comparison table

In [7]:
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision (Survived)', 'Recall (Survived)', 'F1-score (Survived)'],
    'Baseline (default C=1.0, l2)': [
        accuracy_score(y_test, y_pred_baseline),
        precision_score(y_test, y_pred_baseline),
        recall_score(y_test, y_pred_baseline),
        f1_score(y_test, y_pred_baseline),
    ],
    'Tuned (GridSearchCV)': [
        accuracy_score(y_test, y_pred_tuned),
        precision_score(y_test, y_pred_tuned),
        recall_score(y_test, y_pred_tuned),
        f1_score(y_test, y_pred_tuned),
    ],
})
comparison['Change'] = comparison['Tuned (GridSearchCV)'] - comparison['Baseline (default C=1.0, l2)']
comparison = comparison.round(4)
comparison


,Metric,"Baseline (default C=1.0, l2)",Tuned (GridSearchCV),Change
0,Accuracy,0.8101,0.8101,0.0000
1,Precision (Survived),0.7778,0.7692,-0.0085
2,Recall (Survived),0.7101,0.7246,0.0145
3,F1-score (Survived),0.7424,0.7463,0.0038


## 9. Summary — Model Evaluation & Tuning

- Went beyond accuracy: calculated precision, recall, and F1-score per class using `classification_report`.
- Explained why accuracy alone is misleading on imbalanced data — it can hide poor performance on the minority class.
- Tuned `C` (regularization strength) and `penalty` (`l1`/`l2`) using `GridSearchCV` with 5-fold cross-validation, optimizing for F1-score rather than accuracy.
- Compared the tuned model against the original baseline in the before/after table above.
- **Result:** see the table for exact numbers — tuning found the best regularization strength for this feature set; the size of the improvement (or lack of one) shows that a simple Logistic Regression was already close to its ceiling on these features, and further gains would likely need better features (e.g. `family_size`) more than further tuning.
